In [20]:
%pylab
%matplotlib inline
import numpy as np
from os.path import getsize
from dask import delayed
import dask.array as da

ImportError: Cannot load backend 'MacOSX' which requires the 'macosx' interactive framework, as 'qt' is currently running

In [ ]:
filename = 'Image_001_001.raw'
r = open(filename,'rb')
width=1024
height=288
nbytes = getsize(filename)
framesize = width*height*2
nFrames = int(nbytes/framesize)
r.close()

In [ ]:
def getImageFromFilename(filename,n,width,height):
    rawimg = open(filename,'rb')
    frameSize = width*height*2
    offset = n*frameSize
    rawimg.seek(offset)
    st = rawimg.read(frameSize)
    nparray = np.frombuffer(st,dtype = np.uint16).reshape((width,height))
    #r.close()
    return nparray

In [ ]:
def getImage(filename,rawimg,n,width,height):
    frameSize = width*height*2
    offset = n*frameSize
    rawimg.seek(offset)
    st = rawimg.read(frameSize)
    nparray = np.frombuffer(st,dtype = np.uint16).reshape((1,height,width))
    return nparray

In [ ]:
lazy_imread = delayed(getImageFromFilename)

In [ ]:
lazy_arrays = [lazy_imread(filename,n,height,width) for n in range(nFrames)]

In [ ]:
dask_array = [da.from_delayed(delayed_reader,shape=[height,width],dtype=np.uint16) for delayed_reader in lazy_arrays]

In [ ]:
stack = da.stack(dask_array,axis=0)
stack.shape 

(7330, 288, 1024)

In [ ]:
stack.shape

(7330, 288, 1024)

In [ ]:
stack = stack.rechunk()
stack.compute_chunk_sizes()

dask.array<rechunk-merge, shape=(7330, 288, 1024), dtype=uint16, chunksize=(227, 288, 1024), chunktype=numpy.ndarray>

In [ ]:
imshow(stack[0:1,0,:,:].mean(0))

IndexError: Too many indices for array

In [ ]:
import dask_image
import dask_image.ndfilters

In [ ]:
stack.shape

(7330, 288, 1024)

In [ ]:
stack3= dask_image.ndfilters.gaussian_filter(stack,sigma=(2,2,2))

In [ ]:
stack3.shape

(7330, 288, 1024)

In [ ]:
stack3 = stack3.rechunk()

In [ ]:
stack4 = stack3.rechunk({0:1,1:1024,2:288})

In [ ]:
stack3

dask.array<_trim, shape=(7330, 288, 1024), dtype=uint16, chunksize=(227, 288, 1024), chunktype=numpy.ndarray>

In [ ]:
stack3.to_zarr('test9.zarr',compressor=None,overwrite=True)

In [ ]:
import dask.array as da
stack3 = da.from_zarr('test9.zarr')

In [ ]:
stack3.persist()

dask.array<from-zarr, shape=(7330, 288, 1024), dtype=uint16, chunksize=(227, 288, 1024), chunktype=numpy.ndarray>

Traceback (most recent call last):
  File "/Users/federico/miniforge3/envs/napari/lib/python3.10/site-packages/napari/_qt/widgets/qt_dims_slider.py", line 141, in _value_changed
    self.dims.set_current_step(self.axis, value)
  File "/Users/federico/miniforge3/envs/napari/lib/python3.10/site-packages/napari/components/dims.py", line 288, in set_current_step
    self.current_step = full_current_step
  File "/Users/federico/miniforge3/envs/napari/lib/python3.10/site-packages/napari/utils/events/evented_model.py", line 241, in __setattr__
    getattr(self.events, name)(value=after)  # emit event
  File "/Users/federico/miniforge3/envs/napari/lib/python3.10/site-packages/napari/utils/events/event.py", line 757, in __call__
    self._invoke_callback(cb, event if pass_event else None)
  File "/Users/federico/miniforge3/envs/napari/lib/python3.10/site-packages/napari/utils/events/event.py", line 794, in _invoke_callback
    _handle_exception(
  File "/Users/federico/miniforge3/envs/napari/li

In [ ]:
import napari
app = napari.view_image(stack3[:,:,:], contrast_limits=[0,1000], multiscale=False)

Traceback (most recent call last):
  File "/Users/federico/miniforge3/envs/napari/lib/python3.10/site-packages/vispy/app/backends/_qt.py", line 502, in mousePressEvent
    self._vispy_mouse_press(
  File "/Users/federico/miniforge3/envs/napari/lib/python3.10/site-packages/vispy/app/base.py", line 184, in _vispy_mouse_press
    ev = self._vispy_canvas.events.mouse_press(**kwargs)
  File "/Users/federico/miniforge3/envs/napari/lib/python3.10/site-packages/vispy/util/event.py", line 453, in __call__
    self._invoke_callback(cb, event)
  File "/Users/federico/miniforge3/envs/napari/lib/python3.10/site-packages/vispy/util/event.py", line 471, in _invoke_callback
    _handle_exception(self.ignore_callback_errors,
  File "/Users/federico/miniforge3/envs/napari/lib/python3.10/site-packages/vispy/util/event.py", line 469, in _invoke_callback
    cb(event)
  File "/Users/federico/miniforge3/envs/napari/lib/python3.10/site-packages/napari/_qt/qt_viewer.py", line 1067, in on_mouse_press
    self.

In [ ]:
l=app.layers[0]

In [ ]:
app.add_image(stack3[:,:,:])

<Image layer 'stack3' at 0x17728fbb0>

In [ ]:
napari.close('all')

AttributeError: No napari attribute close

In [1]:
%pylab
%matplotlib inline
import numpy as np
from os.path import getsize
from dask import delayed
import dask.array as da

from scipy.ndimage import gaussian_filter
filename = 'Image_001_001.raw'
r = open(filename,'rb')
width=1024
height=288
nbytes = getsize(filename)
framesize = width*height*2
nFrames = int(nbytes/framesize)
r.close()

Using matplotlib backend: <object object at 0x1035e64c0>
%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib


In [2]:
def getImage(rawimg,n,width,height):
    frameSize = width*height*2
    offset = n*frameSize
    rawimg.seek(offset)
    st = rawimg.read(frameSize)
    nparray = np.frombuffer(st,dtype = np.uint16).reshape((1,height,width))
    return nparray

def loadWholeStack(filename,width,height,start=0,end=-1,step=1):
    r = open(filename,'rb')
    nbytes = getsize(filename)
    framesize = width*height*2
    nFrames = int(nbytes/framesize)
    
    if end == -1:
        end = nFrames
    totalSize = end-start

    stack = np.zeros((height,width,totalSize),dtype=np.uint16)
    for i in range(start,end,step):
        stack[:,:,i-start] = gaussian_filter(getImage(r,i,width,height),2)
        #stack.append(getImage(r,i,width,height))
    stack = stack.swapaxes(0,2)
    stack = stack.swapaxes(1,2)


    for i in range(stack.shape[1]):
        stack[:,i,:] = gaussian_filter(stack[:,i,:],(2,0))

    r.close()
    return stack

In [3]:
r = loadWholeStack(filename,width,height,start=0,end=200)


In [4]:
import napari
app = napari.view_image(r, contrast_limits=[0,1000], multiscale=False)

In [6]:
app.add_image(r)

TypeError: ViewerModel.add_image() got an unexpected keyword argument 'layer'

In [10]:
l = app.layers['r']

In [13]:
l.data = r

In [19]:
r2 = loadWholeStack(filename,width,height,start=0,end=1200)

In [20]:
r3= vstack((r3,r2))

In [21]:
l.data = r3

In [22]:
r = loadWholeStack(filename,width,height)

In [24]:
l.data = r

Traceback (most recent call last):
  File "/Users/federico/miniforge3/envs/napari/lib/python3.10/site-packages/napari/utils/action_manager.py", line 227, in <lambda>
    button.clicked.connect(lambda: self.trigger(name))
  File "/Users/federico/miniforge3/envs/napari/lib/python3.10/site-packages/napari/utils/action_manager.py", line 427, in trigger
    return self._actions[name].injected()
  File "/Users/federico/miniforge3/envs/napari/lib/python3.10/site-packages/in_n_out/_store.py", line 773, in _exec
    result = func(**{**_kwargs, **bound.arguments})
  File "/Users/federico/miniforge3/envs/napari/lib/python3.10/site-packages/napari/layers/utils/layer_utils.py", line 125, in _wrapper
    obj = kwargs[first_variable_name]
KeyError: 'layer'
Traceback (most recent call last):
  File "/Users/federico/miniforge3/envs/napari/lib/python3.10/site-packages/in_n_out/_store.py", line 773, in _exec
    result = func(**{**_kwargs, **bound.arguments})
TypeError: _toggle_visibility() missing 1 req

In [19]:
import os
from scipy.ndimage import gaussian_filter
import numpy as np
from os.path import getsize
import napari
from skimage.io import imread

filename =  'Image_001_001.raw'
previewFilename = 'ChanC_Preview.tif'

class thorlabsFile():
   
    def __init__(self,folder) -> None:

        self.folder = folder
        self.fullpath = os.path.join(self.folder,filename)

        prev = imread(os.path.join('./',previewFilename))

        self.width = prev.shape[1] #change with self determination
        self.height = prev.shape[0]

        self.r = open(self.fullpath,'rb')
        nbytes = getsize(self.fullpath)
        self.frameSize = self.width*self.height*2
        self.nFrames = int(nbytes/self.frameSize)

        self.currentLastFrame = 0

        self.array = np.empty((0,self.height,self.width))
        self.app = napari.Viewer()
        #self.app.add_image(self.array)

    def loadFile(self,folder):

        try:
            self.r.close()
        except:
            pass
        
        self.folder = folder
        self.fullpath = os.path.join(self.folder,filename)
        prev = imread(os.path.join('./',previewFilename))

        self.width = prev.shape[1] #change with self determination
        self.height = prev.shape[0]

        self.r = open(self.fullpath,'rb')
        nbytes = getsize(self.fullpath)
        self.frameSize = self.width*self.height*2
        self.nFrames = int(nbytes/self.frameSize)

        self.currentLastFrame = 0

        self.array = np.empty((0,self.height,self.width))

        l = self.app.layers['Image']
        l.data = self.array
        
    def getImage(self,n):

        offset = n*self.frameSize
        self.r.seek(offset)
        st = self.r.read(self.frameSize)
        nparray = np.frombuffer(st,dtype = np.uint16).reshape((1,self.height,self.width))
        
        return nparray

    def loadWholeStack(self,start=0,end=-1,step=1):

        
        if end == -1:
            end = nFrames
        totalSize = end-start

        #stack = np.zeros((self.height,self.width,totalSize),dtype=np.uint16)
        #for i in range(start,end,step):
        #    stack[:,:,i-start] = gaussian_filter(self.getImage(i),2)
        #    #stack.append(getImage(r,i,width,height))
        
        offset = start*self.frameSize
        self.r.seek(offset)
        st = self.r.read(self.frameSize*totalSize)
        stack = np.frombuffer(st,dtype = np.uint16).reshape((totalSize,self.height,self.width))
        stack2 = stack.copy()
        stack2.setflags(write=1)

      #  stack = stack.swapaxes(0,2)
      #  stack = stack.swapaxes(1,2)
        

        #for i in range(stack2.shape[1]):
        #    stack2[:,i,:] = gaussian_filter(stack2[:,i,:],(2,0))
        stack2 = gaussian_filter(stack2,(2,2,2))
        return stack2

    def loadNextNFrames(self,n):

        self.array= np.vstack([self.array, self.loadWholeStack(start=self.currentLastFrame,end = self.currentLastFrame+n)])

        try:
            l = self.app.layers['Image']
            l.data = self.array
        except:
            self.app.add_image(self.array)

        self.currentLastFrame = self.array.shape[0]

    def loadUpToFrameN(self,n):
        
        if (n<self.currentLastFrame) | (n>self.nFrames):
            return

        self.array= np.vstack([self.array, self.loadWholeStack(start=self.currentLastFrame,end = n)])

        if self.currentLastFrame == 0:
                self.app.add_image(self.array)
        else:
            l = self.app.layers['Image']
            l.data = self.array
        self.currentLastFrame = self.array.shape[0]






In [20]:

tb = thorlabsFile('./')

In [80]:
tb.loadFile('./') 

In [21]:
tb.loadNextNFrames(1000)

In [6]:
tb.loadUpToFrameN(5000)

In [19]:
app.close()

NameError: name 'app' is not defined

Traceback (most recent call last):
  File "/Users/federico/miniforge3/envs/napari/lib/python3.10/site-packages/napari_accelerated_pixel_and_object_classification/_dock_widget.py", line 235, in update_layer
    self.update_memory_consumption()
  File "/Users/federico/miniforge3/envs/napari/lib/python3.10/site-packages/napari_accelerated_pixel_and_object_classification/_dock_widget.py", line 338, in update_memory_consumption
    number_of_pixels = np.sum(tuple([np.prod(i.shape) for i in self.get_selected_images_data()]))
  File "/Users/federico/miniforge3/envs/napari/lib/python3.10/site-packages/napari_accelerated_pixel_and_object_classification/_dock_widget.py", line 422, in get_selected_images_data
    image_layers = self.get_selected_images()
  File "/Users/federico/miniforge3/envs/napari/lib/python3.10/site-packages/napari_accelerated_pixel_and_object_classification/_dock_widget.py", line 416, in get_selected_images
    item = self.image_list.item(i)
RuntimeError: wrapped C/C++ objec

In [18]:
prev.shape

(288, 1024)